In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Fig. 5: the centre of an LMC-mass halo in cold and self-interacting dark matter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/letter/fig5_sidm.ipynb)

Every number in the text and caption is written by `scripts/paper_text_letter.py` from the result tables
(`results/macros_letter.tex`); the code cells are the figure script itself (`scripts/paper_sims_sidm.py`), step by step.

**What it shows.** Figure 5 compares the centre of one halo with the mass of the Large Magellanic Cloud's (LMC's) halo,
simulated with cold dark matter (CDM) and with self-interacting dark matter (SIDM), in which dark-matter particles scatter off each
other. Both runs are of Halo104 of SIDM Concerto ([Nadler et al. 2025](https://doi.org/10.3847/1538-4357/adf553)), from the same
initial conditions and with dark matter only; the SIDM cross-section per unit mass has a normalisation $\sigma_0/m$ =
147 cm² g⁻¹ and falls as $v^{-4}$ with the collision speed $v$ above $w$ = 120 km/s. The
halo's shape is measured in two ways. The axis ratio $c/a$ of its minor to its major axis is found in ellipsoidal shells. The
quadrupole of its gravitational potential is measured relative to the monopole: $\epsilon_{2,\rm all}=|\Phi_2|/|\Phi_0|$ includes
all of the quadrupole, also its part axisymmetric about the halo's minor axis, and $\epsilon_2$ only the part not axisymmetric
about that axis. (In Fig. 4 of the Letter $\epsilon_2$ is taken about the disc's spin; the full quadrupole is the same about any
axis and bounds the non-axisymmetric part about every axis.) With self-interactions the centre of the halo is nearly round already
at redshift $z\approx$ 4, with a quadrupole at or below the one at which the TNG50 cohorts keep their
rotation; in CDM it stays flattened and aspherical.

**Starts from:** the quick route. The high-redshift snapshots behind `results/sims_sidm_shapes.csv` are not public; they are
available from the simulation authors on request (`scripts/fetch_concerto.py` explains how to fetch them once access is given).

## Making the figure, step by step

<details><summary>About <code>scripts/paper_sims_sidm.py</code></summary>

```text
Paper figure (Section 6.6, Fig. sims_sidm): the centre of an LMC-mass halo in cold and self-interacting dark matter, against the TNG50
progenitors.

Inputs: results/sims_sidm_shapes.csv (scripts/sims_sidm_shapes.py: Concerto Halo104, CDM and GroupSIDM, z = 4, 3, 2, 1.5, 0) and
results/sims_halo_settle.csv (scripts/sims_halo_settle.py: TNG50, 60 progenitors).
(a) eps2 at 2 kpc against redshift: Concerto CDM and SIDM, all of l = 2 (filled)
and the part not axisymmetric about the minor axis of the 1.5-2.5 kpc shell (open); TNG50 DM, the part not axisymmetric about the
stellar disc axis (median and 16th-84th percentiles over the progenitors of the per-galaxy median in redshift bins of 0.5, z >= 1,
bins with >= 20 galaxies).
(b) c/a of the host in ellipsoidal shells against the shell's major-axis radius (shells with >= NMIN particles); (c) eps2 =
|Phi_2|/|Phi_0| of the potential against radius. In (b) and (c) CDM in greys and SIDM in reds, light at z = 4 to dark at z = 0.
Writes figures/paper_sims_sidm.png; prints the TNG50 bins.
Run: .venv/bin/python scripts/manifest.py run paper_sims_sidm.py   (after scripts/sims_sidm_shapes.py and sims_halo_settle.py; 1 s in
the recorded run of docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_sims_sidm.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/sims_halo_settle.csv` | repository | written by `python scripts/sims_halo_settle.py` |
| `results/sims_sidm_shapes.csv` | repository | written by `python scripts/sims_sidm_shapes.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple

from lmcold.layout import panel_grid
from lmcold.sources import ROOT

### Load the data
The figure reads two result tables and no simulation; the simulations are needed only to remake them:
`results/sims_sidm_shapes.csv` (`scripts/sims_sidm_shapes.py`, from the Concerto Halo104 runs in cold and self-interacting dark
matter) and `results/sims_halo_settle.csv` (`scripts/sims_halo_settle.py`, from the dark matter of 60 TNG50 progenitors).

In [ ]:
S = pd.read_csv(ROOT / 'results/sims_sidm_shapes.csv')   # Concerto Halo104, CDM and SIDM: shell axis ratios and eps2 at five redshifts
T = pd.read_csv(ROOT / 'results/sims_halo_settle.csv', dtype={'halo': str})   # TNG50 progenitors: eps2 at 2 kpc per snapshot

### The Concerto halo and the TNG50 band
`S`: axis ratios and $\epsilon_2$ of Halo104 in ellipsoidal shells, CDM and SIDM, at five redshifts (rows with `N = 0` are the values
at 2 kpc). The TNG50 band is $\epsilon_2$ of the progenitors (`T`) in redshift bins.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
NMIN = 10000   # particles per shell for a shape to be drawn in (b) and (c)
Z_BIN, MIN_GAL, Z_MIN = 0.5, 20, 1.0   # TNG50 band: redshift bins, galaxies per bin, lowest redshift

T['zb'] = (T.z / Z_BIN).round() * Z_BIN
g = T.groupby(['zb', 'halo']).eps2.median().groupby(level=0)
tng = pd.DataFrame({'m': g.median(), 'lo': g.quantile(.16), 'hi': g.quantile(.84), 'n': g.size()})
tng = tng[(tng.n >= MIN_GAL) & (tng.index >= Z_MIN)]

### The figure
(a) $\epsilon_2$ at 2 kpc against redshift; (b) $c/a$ and (c) $\epsilon_{2,\rm all}$ against radius, light to dark from $z=4$ to 0.

In [ ]:
fig, ax = panel_grid(1, 3, width=7.0, left=0.5, right=0.1, hgap=0.62, top=0.3)
ax = ax[0]
zs = sorted(S.z.unique(), reverse=True)
shade = np.linspace(0.35, 1.0, len(zs))   # light at z = 4 to dark at z = 0
COL = {'CDM': dict(zip(zs, plt.cm.Greys(shade))), 'SIDM': dict(zip(zs, plt.cm.Reds(0.3 + 0.65 * (shade - 0.35) / 0.65)))}
for z in zs:
    for run in ('CDM', 'SIDM'):
        w = S[(S.z == z) & (S.run == run) & (S.N >= NMIN)]
        ax[1].plot(w.r, w.s, '-', marker='o', ms=2, lw=1, c=COL[run][z])
        ax[2].plot(w.r, w.eps2, '-', marker='o', ms=2, lw=1, c=COL[run][z])
zh = [tuple(plt.Line2D([], [], c=COL[run][z], lw=1.5) for run in ('CDM', 'SIDM')) for z in zs]
ax[2].legend(zh, [f'$z={z:.1f}$' for z in zs], handler_map={tuple: HandlerTuple(ndivide=None, pad=0)}, frameon=False, fontsize=5, ncol=2,
             loc='lower right', handlelength=2.4, columnspacing=0.7, handletextpad=0.4)   # empty corner (PI)
ax[1].set(xscale='log', ylim=(0, 1.02), xlabel='major-axis radius [kpc]', ylabel='minor-to-major axis ratio $c/a$')
ax[2].set(xscale='log', yscale='log', xlabel='radius [kpc]', ylabel=r'$\epsilon_{2,\rm all}=|\Phi_2|/|\Phi_0|$')
ax[0].fill_between(tng.index, tng.lo, tng.hi, color='0.5', alpha=0.25, lw=0)
ax[0].plot(tng.index, tng.m, '-', c='0.4', lw=1.5, label='TNG50 (with baryons)')
for run, c in (('CDM', 'k'), ('SIDM', 'C3')):
    e = S[(S.run == run) & (S.N == 0)].sort_values('z')
    ax[0].plot(e.z, e.eps2, '-o', c=c, ms=3, lw=1, label=f'Concerto {run}')
    ax[0].plot(e.z, e.eps2_na, ':o', c=c, ms=3, lw=0.8, mfc='white')
ax[0].set(yscale='log', xlim=(4.2, -0.2), ylim=(1.5e-4, 0.06), xlabel='redshift', ylabel=r'$\epsilon_{2,\rm all}$ (filled), $\epsilon_2$ (open) at 2 kpc')
ax[0].legend(frameon=False, fontsize=5, loc='lower left', handlelength=1.8, handletextpad=0.4)   # empty corner (PI)
for a, t in zip(ax, ('(a) quadrupole at 2 kpc', '(b) shape of the halo', '(c) its potential quadrupole')):
    a.set_title(t, loc='left', fontsize=8)
fig.savefig(ROOT / 'figures/paper_sims_sidm.png', dpi=300)
print(tng.round(4).to_string())

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_sims_sidm.png')

**Figure 5.** The centre of an LMC-mass halo in cold (CDM) and self-interacting (SIDM) dark matter (SIDM Concerto
Halo104, dark matter only). (a) Quadrupole at 2 kpc against redshift: CDM (black) and SIDM (red), the full
quadrupole including its axisymmetric part ($\epsilon_{2,\rm all}$, filled) and the part not axisymmetric about the halo's minor
axis ($\epsilon_2$, open); grey: TNG50. (b) Minor-to-major axis ratio $c/a$ and (c) $\epsilon_{2,\rm all}$ against radius, light
to dark from $z$ = 4 to 0. With self-interactions the centre of the halo is round already at
$z\approx$ 4; in CDM it stays flattened and aspherical.

## How the inputs are made

The figure script reads the shapes of the Concerto halo (`results/sims_sidm_shapes.csv`) and of the TNG50 progenitors
(`results/sims_halo_settle.csv`). The logged steps upstream of the figure, nearest first:

In [ ]:
import kbtools as kb
kb.upstream('paper_sims_sidm.png')

`python scripts/sims_sidm_shapes.py` measures the axis ratios and the quadrupole of both runs at each redshift (about 15 min; AGAMA;
about 20 GB of memory at z = 0); its code is shown step by step, not run, in the chapter 'How the simulated old stars lose their
rotation'.